# v3.0.0 Dementia PPG AST — Foundation Model Comparison

PPG-based dementia screening using the same AST architecture.
Compares phonetic representations from a speech foundation model vs mel-spectrograms.

In [ ]:
#1 - Imports & paths
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from tqdm import tqdm

ROOT = Path('/data0/b2ai-voice/3.0.0')
PPG_PATH = ROOT / 'features' / 'ppgs.parquet'
DEM_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'cognitive_impairment.tsv'
CTRL_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'control.tsv'
RESULTS_DIR = Path('/home/saptpurk/bridge2ai-voice-parkinsons-ast/results/v3')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_TASKS = [
    'prolonged-vowel', 'glides-high-to-low', 'glides-low-to-high',
    'diadochokinesis-pataka', 'rainbow-passage', 'picture-description',
    'story-recall', 'maximum-phonation-time-1',
]
MIN_TIME_FRAMES = 100

In [ ]:
#2 - Load PPG data
pf = pq.ParquetFile(PPG_PATH)
parts = []
for i in range(pf.num_row_groups):
    parts.append(pf.read_row_group(i, columns=['participant_id','session_id','task_name','ppgs','n_frames']).to_pandas())
ppg_df = pd.concat(parts, ignore_index=True)
ppg_df['participant_id'] = ppg_df['participant_id'].astype(str).str.zfill(6)
print(f'Total PPG recordings: {len(ppg_df)}')

In [ ]:
#3 - Build dementia labels
dem_df = pd.read_csv(DEM_PHEN, sep='\t')
ctrl_df = pd.read_csv(CTRL_PHEN, sep='\t')
dem_ids = set(dem_df['participant_id'].astype(str).str.zfill(6))
ctrl_ids = set(ctrl_df['participant_id'].astype(str).str.zfill(6))
ctrl_ids_clean = ctrl_ids - dem_ids

ppg_df['label'] = np.nan
ppg_df.loc[ppg_df['participant_id'].isin(dem_ids), 'label'] = 1
ppg_df.loc[ppg_df['participant_id'].isin(ctrl_ids_clean), 'label'] = 0

data = ppg_df.dropna(subset=['label']).copy()
data['label'] = data['label'].astype(int)
data = data[(data['task_name'].isin(SELECTED_TASKS)) & (data['n_frames'] >= MIN_TIME_FRAMES)].copy()

print(f'Dementia recordings: {(data["label"]==1).sum()} from {data[data["label"]==1]["participant_id"].nunique()} ppl')
print(f'Control recordings: {(data["label"]==0).sum()} from {data[data["label"]==0]["participant_id"].nunique()} ppl')

In [ ]:
#4 - Process PPGs: (40, T) -> pad/crop to 1024 -> resize to (128, 1024)
from scipy.ndimage import zoom

TARGET_SEQ_LEN = 1024

def process_ppg(ppg_raw, target_len=1024):
    spec = np.stack(ppg_raw).astype(np.float32)
    n_phonemes, time_len = spec.shape
    if time_len < target_len:
        pad_width = target_len - time_len
        spec = np.pad(spec, ((0, 0), (0, pad_width)), mode='reflect')
    elif time_len > target_len:
        start = (time_len - target_len) // 2
        spec = spec[:, start:start + target_len]
    return spec

def resize_ppg(spec, target_mel=128, target_time=1024):
    mel_ratio = target_mel / spec.shape[0]
    time_ratio = target_time / spec.shape[1]
    return zoom(spec, (mel_ratio, time_ratio), order=1).astype(np.float32)

X_list = []
for idx, row in tqdm(data.iterrows(), total=len(data), desc='Processing PPGs'):
    processed = process_ppg(row['ppgs'], TARGET_SEQ_LEN)
    X_list.append(processed)

X_raw = np.stack(X_list)
y_raw = data['label'].values
participants_raw = data['participant_id'].values
print(f'Processed shape: {X_raw.shape}')

In [ ]:
#5 - Model and helpers (identical to spectrogram pipeline)
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import ASTModel, ASTConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class ASTClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, freeze_base=False):
        super().__init__()
        if pretrained:
            self.ast = ASTModel.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
            hidden_size = self.ast.config.hidden_size
        else:
            config = ASTConfig(hidden_size=768, num_hidden_layers=12, num_attention_heads=12,
                             intermediate_size=3072, max_length=1024, num_mel_bins=128)
            self.ast = ASTModel(config)
            hidden_size = 768
        if freeze_base:
            for param in self.ast.parameters():
                param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size), nn.Linear(hidden_size, 256),
            nn.GELU(), nn.Dropout(0.3), nn.Linear(256, num_classes))
    def forward(self, x):
        x = x.transpose(1, 2)
        outputs = self.ast(input_values=x)
        return self.classifier(outputs.pooler_output)

class ASTDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, participants, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.participants = np.array(participants)
        self.augment = augment
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            if np.random.random() < 0.5:
                t = np.random.randint(50, 150)
                t0 = np.random.randint(0, max(1, x.shape[1] - t))
                x[:, t0:t0+t] = 0
            if np.random.random() < 0.5:
                f = np.random.randint(10, 30)
                f0 = np.random.randint(0, max(1, x.shape[0] - f))
                x[f0:f0+f, :] = 0
        return {'inputs': x, 'labels': self.y[idx], 'participant': self.participants[idx]}

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

def evaluate_fold(model, loader, device):
    model.eval()
    all_probs, all_labels, all_parts = [], [], []
    with torch.no_grad():
        for batch in loader:
            inputs = batch['inputs'].to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(batch['labels'].numpy())
            all_parts.extend(batch['participant'])
    all_probs, all_labels, all_parts = np.array(all_probs), np.array(all_labels), np.array(all_parts)
    unique_parts = np.unique(all_parts)
    part_probs = np.array([all_probs[all_parts == p].mean() for p in unique_parts])
    part_labels = np.array([all_labels[all_parts == p][0] for p in unique_parts])
    return part_probs, part_labels, unique_parts

In [ ]:
#6 - 5-Fold Cross-Validation (identical to spectrogram pipeline)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve
from sklearn.metrics import confusion_matrix, accuracy_score
from scipy import stats
import copy, time

unique_participants = np.unique(participants_raw)
participant_labels = np.array([y_raw[participants_raw == p][0] for p in unique_participants])

print(f'Total participants: {len(unique_participants)}')
print(f'Dementia: {int(participant_labels.sum())}, Control: {int((participant_labels == 0).sum())}')

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []
all_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
all_oof_labels = participant_labels.astype(np.int64).copy()
norm_stats = {}
total_start = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(unique_participants, participant_labels)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
    train_parts_fold = unique_participants[train_idx]
    val_parts_fold = unique_participants[val_idx]
    train_mask = np.isin(participants_raw, train_parts_fold)
    val_mask = np.isin(participants_raw, val_parts_fold)

    X_train_fold = X_raw[train_mask]; y_train_fold = y_raw[train_mask]; parts_train_fold = participants_raw[train_mask]
    X_val_fold = X_raw[val_mask]; y_val_fold = y_raw[val_mask]; parts_val_fold = participants_raw[val_mask]
    print(f'Train: {len(X_train_fold)} recs from {len(train_parts_fold)} ppl, Val: {len(X_val_fold)} recs from {len(val_parts_fold)} ppl')

    # Resize 40 -> 128
    X_train_ast = np.stack([resize_ppg(x) for x in tqdm(X_train_fold, desc='resize train', leave=False)])
    X_val_ast = np.stack([resize_ppg(x) for x in tqdm(X_val_fold, desc='resize val', leave=False)])

    fold_mean = X_train_ast.mean(); fold_std = X_train_ast.std()
    X_train_ast = (X_train_ast - fold_mean) / (fold_std + 1e-8)
    X_val_ast = (X_val_ast - fold_mean) / (fold_std + 1e-8)
    norm_stats[fold + 1] = {'mean': float(fold_mean), 'std': float(fold_std)}

    train_ds = ASTDataset(X_train_ast, y_train_fold, parts_train_fold, augment=True)
    val_ds = ASTDataset(X_val_ast, y_val_fold, parts_val_fold, augment=False)
    cc = np.bincount(y_train_fold); weights = 1.0 / cc; sample_weights = weights[y_train_fold]
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights))
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, sampler=sampler, num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    model = ASTClassifier(num_classes=2, pretrained=True, freeze_base=False).to(device)
    backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
    head_params = [p for n, p in model.named_parameters() if 'classifier' in n]
    optimizer = torch.optim.AdamW([{'params': backbone_params, 'lr': 5e-6, 'weight_decay': 0.01},
                                   {'params': head_params, 'lr': 5e-4, 'weight_decay': 0.01}], betas=(0.9, 0.999))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-7)
    cw = (cc.sum() / (2.0 * cc)).astype(np.float32)
    criterion = FocalLoss(alpha=torch.tensor(cw, dtype=torch.float32).to(device), gamma=2.0)

    best_score, best_state, patience_counter = 0, None, 0
    for epoch in range(30):
        model.train(); total_loss = 0
        for batch in train_loader:
            inputs = batch['inputs'].to(device); labels = batch['labels'].to(device)
            optimizer.zero_grad(); outputs = model(inputs); loss = criterion(outputs, labels)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        part_probs, part_labels_fold, _ = evaluate_fold(model, val_loader, device)
        if len(np.unique(part_labels_fold)) > 1:
            auc = roc_auc_score(part_labels_fold, part_probs)
            fpr, tpr, thresholds = roc_curve(part_labels_fold, part_probs)
            opt_thresh = thresholds[np.argmax(tpr - fpr)]
            f1_opt = f1_score(part_labels_fold, (part_probs >= opt_thresh).astype(int), zero_division=0)
        else: auc, f1_opt = 0.5, 0.0
        score = 0.4 * auc + 0.6 * f1_opt
        if score > best_score + 0.01:
            best_score = score; best_state = copy.deepcopy(model.state_dict()); patience_counter = 0
        else: patience_counter += 1
        if patience_counter >= 10: print(f'  Early stopping at epoch {epoch+1}'); break

    model.load_state_dict(best_state)
    part_probs, part_labels_fold, val_pids = evaluate_fold(model, val_loader, device)
    torch.save(model.state_dict(), str(RESULTS_DIR / f'ppg_ast_dementia_v3_fold{fold+1}.pt'))
    for i, pid in enumerate(val_pids):
        idx_oof = np.where(unique_participants == pid)[0][0]
        all_oof_probs[idx_oof] = part_probs[i]

    fold_auc = roc_auc_score(part_labels_fold, part_probs)
    fpr, tpr, thresholds = roc_curve(part_labels_fold, part_probs)
    opt_thresh = thresholds[np.argmax(tpr - fpr)]
    preds_opt = (part_probs >= opt_thresh).astype(int)
    fold_results.append({'fold': fold+1, 'auc': float(fold_auc),
        'f1_opt': float(f1_score(part_labels_fold, preds_opt, zero_division=0)),
        'recall_opt': float(recall_score(part_labels_fold, preds_opt, zero_division=0)),
        'precision_opt': float(precision_score(part_labels_fold, preds_opt, zero_division=0)),
        'threshold': float(opt_thresh)})
    print(f'Fold {fold+1}: AUC={fold_auc:.4f}')
    del model, optimizer; torch.cuda.empty_cache()

print(f'\nTotal CV time: {(time.time()-total_start)/60:.1f} min')

In [ ]:
#7 - Summary and save
aucs = [r['auc'] for r in fold_results]
f1s = [r['f1_opt'] for r in fold_results]
recalls = [r['recall_opt'] for r in fold_results]
precisions = [r['precision_opt'] for r in fold_results]

t_crit = stats.t.ppf(0.975, df=4)
print('Per-fold results:')
for r in fold_results:
    print(f'  Fold {r["fold"]}: AUC={r["auc"]:.4f} F1={r["f1_opt"]:.4f}')
print('\nMean +/- SD [95% CI]:')
for name, arr in [('AUC', aucs), ('F1', f1s), ('Recall', recalls), ('Precision', precisions)]:
    m, sd = np.mean(arr), np.std(arr, ddof=1)
    se = sd / np.sqrt(5)
    print(f'  {name}: {m:.4f} ({sd:.4f}) [{m-t_crit*se:.4f}, {m+t_crit*se:.4f}]')

oof_auc = roc_auc_score(all_oof_labels, all_oof_probs)
fpr, tpr, thresholds = roc_curve(all_oof_labels, all_oof_probs)
oof_thresh = thresholds[np.argmax(tpr - fpr)]
oof_preds = (all_oof_probs >= oof_thresh).astype(int)
print(f'\nOOF AUC: {oof_auc:.4f}')
print(f'OOF F1:  {f1_score(all_oof_labels, oof_preds, zero_division=0):.4f}')

np.savez(str(RESULTS_DIR / 'ppg_ast_dementia_v3_cv_results.npz'),
    oof_probs=all_oof_probs, oof_labels=all_oof_labels, participant_ids=unique_participants,
    fold_aucs=np.array(aucs), fold_f1s=np.array(f1s),
    fold_recalls=np.array(recalls), fold_precisions=np.array(precisions),
    fold_thresholds=np.array([r['threshold'] for r in fold_results]),
    oof_auc=np.array(oof_auc), oof_thresh=np.array(oof_thresh),
    norm_means=np.array([norm_stats[f+1]['mean'] for f in range(N_FOLDS)]),
    norm_stds=np.array([norm_stats[f+1]['std'] for f in range(N_FOLDS)]))
print(f'Saved: {RESULTS_DIR}/ppg_ast_dementia_v3_cv_results.npz')